In [1]:
# ============================================================
# SIG-04 AUTOMATRIX
# Live Sensor + TinyML
# LDR + ESP32 + OLED
# Classes: Dark, Normal, Bright
# ============================================================

import tensorflow as tf
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

# ============================================================
# 1. Create LDR Training Data
# ============================================================
# LDR ADC range: 0 - 4095
#
# 0 - 1200   -> Dark
# 1201-2800  -> Normal
# 2801-4095  -> Bright

np.random.seed(42)

dark = np.random.randint(100, 1200, 100)
normal = np.random.randint(1201, 2800, 100)
bright = np.random.randint(2801, 4095, 100)

X = np.concatenate([dark, normal, bright]).astype(np.float32)
y = np.array(
    [0] * 100 +
    [1] * 100 +
    [2] * 100
)

# Normalize LDR values
X = X / 4095.0

# Reshape input
X = X.reshape(-1, 1)

# ============================================================
# 2. Create TinyML Model
# ============================================================

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 3. Train Model
# ============================================================

model.fit(
    X,
    y,
    epochs=50,
    batch_size=16,
    verbose=0
)

print("\nModel training completed!")

# ============================================================
# 4. Check Model Accuracy
# ============================================================

loss, accuracy = model.evaluate(X, y, verbose=0)

print("Training Accuracy:",
      round(accuracy * 100, 2), "%")

# ============================================================
# 5. Test Sample LDR Values
# ============================================================

classes = ["Dark", "Normal", "Bright"]

test_values = [500, 2000, 3500]

print("\nSample Predictions:")

for value in test_values:

    sample = np.array([[value / 4095.0]],
                      dtype=np.float32)

    prediction = model.predict(
        sample,
        verbose=0
    )

    predicted_class = np.argmax(prediction[0])

    confidence = prediction[0][predicted_class]

    print(
        "LDR Value:",
        value,
        "->",
        classes[predicted_class],
        "| Confidence:",
        round(float(confidence), 3)
    )

# ============================================================
# 6. Convert TensorFlow Model to TFLite
# ============================================================

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("light_model.tflite", "wb") as f:
    f.write(tflite_model)

print("\nlight_model.tflite created successfully!")

# ============================================================
# 7. Convert TFLite Model to C Header
# ============================================================

os.system(
    "xxd -i light_model.tflite > light_model_data.h"
)

print("light_model_data.h created successfully!")

# ============================================================
# 8. Check Generated Files
# ============================================================

print("\nGenerated Files:")

print(
    "light_model.tflite:",
    os.path.getsize("light_model.tflite"),
    "bytes"
)

print(
    "light_model_data.h:",
    os.path.getsize("light_model_data.h"),
    "bytes"
)

# ============================================================
# 9. Download Files
# ============================================================

from google.colab import files

print("\nDownloading TFLite model...")
files.download("light_model.tflite")

TensorFlow version: 2.21.0

Model training completed!
Training Accuracy: 77.67 %

Sample Predictions:
LDR Value: 500 -> Dark | Confidence: 0.446
LDR Value: 2000 -> Bright | Confidence: 0.379
LDR Value: 3500 -> Bright | Confidence: 0.6
Saved artifact at '/tmp/tmp4hnl2na9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139187998667984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139187998668944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139187998668560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139187998669328: TensorSpec(shape=(), dtype=tf.resource, name=None)

light_model.tflite created successfully!
light_model_data.h created successfully!

Generated Files:
light_model.tflite: 1928 bytes
light_model_data.h: 0 bytes



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>